# Pipeline de Limpieza — Defunciones Generales Ecuador 2021

Proyecto 0.2 — EDA desde Cero con Datos Sucios
Portafolio: De Matemático a Data Scientist

---

## Objetivo

Transformar el dataset crudo de 107,648 registros × 45 columnas en un dataset limpio 
y confiable, aplicando un pipeline de limpieza reproducible que resuelve los 9 problemas 
de calidad identificados en el Notebook 01 (Auditoría de Calidad).

## Problemas a resolver

| # | Problema | Magnitud | Severidad |
|---|---|---|---|
| 1 | Nulos disfrazados: espacios en blanco | 313,117 celdas | Alta |
| 2 | Nulos disfrazados: "Sin información" | 56,186 celdas | Alta |
| 3 | Nulos disfrazados: "No aplica" | 382 celdas | Baja |
| 4 | Fechas imposibles: `9999-99-99` en `fecha_nac` | 103 registros | Media |
| 5 | Fechas parciales: `año-99-99` en `fecha_nac` | 98 registros | Media |
| 6 | Columnas numéricas como texto: `anio_nac`, `dia_nac`, `edad` | 3 columnas | Alta |
| 7 | Registros con año de fallecimiento ≠ 2021 | 2,400 registros | Media |
| 8 | Código + descripción pegados en columnas de causa | 6 columnas | Media |
| 9 | Duplicados exactos (ignorando `Numeracion`) | 7 pares (14 filas) | Baja |

**Nulos estructurales** (NO son errores como habíamos mencionado): muj_fertil, mor_viol, lug_viol tienen 
>90% de nulos porque solo aplican a subconjuntos específicos. Se documentan pero no se 
"corrigen".

## Analogía matemática para el principio de diseño

Cada transformación es una función pura $T_i: \mathcal{D} \to \mathcal{D}$ que recibe 
un DataFrame y devuelve un DataFrame modificado. Al final, todas se componen en:

$$\text{limpiar\_dataset} = T_n \circ T_{n-1} \circ \cdots \circ T_2 \circ T_1$$

donde el orden de composición respeta las dependencias entre transformaciones.

In [1]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuración de visualización
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Rutas relativas
RAW_PATH = Path('../data/raw/')
PROCESSED_PATH = Path('../data/processed/')

# Cargar dataset crudo con los parámetros correctos
df_raw = pd.read_csv(
    RAW_PATH / 'EDG_2021_act_CSV.csv',
    sep=';',
    low_memory=False
)

print(f"Dataset cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")

Dataset cargado: 107,648 filas × 45 columnas


In [2]:
# Guardar métricas del dataset crudo para comparar al final
estado_inicial = {
    'filas': df_raw.shape[0],
    'columnas': df_raw.shape[1],
    'nulos_reales': df_raw.isnull().sum().sum(),
    'duplicados_exactos': df_raw.drop(columns='Numeracion').duplicated().sum(),
    'tipos_texto': (df_raw.dtypes == 'str').sum(),
    'memoria_mb': df_raw.memory_usage(deep=True).sum() / 1_048_576
}

print("Estado inicial del dataset:")
for clave, valor in estado_inicial.items():
    if clave == 'memoria_mb':
        print(f"  {clave}: {valor:.2f} MB")
    else:
        print(f"  {clave}: {valor:,}")

Estado inicial del dataset:
  filas: 107,648
  columnas: 45
  nulos_reales: 0
  duplicados_exactos: 7
  tipos_texto: 42
  memoria_mb: 336.30 MB


## 2. Estandarización de nulos disfrazados (el $T_1$)

**Problemas que resuelve:** #1 (espacios en blanco — 313,117 celdas), #2 ("Sin 
información" — 56,186 celdas) y #3 ("No aplica" — 382 celdas).

**Estrategia:** Definir una lista explícita de valores que representan ausencia de dato 
y reemplazarlos todos por `NaN`. Se aplica `str.strip()` primero para capturar variantes 
con espacios antes o después del texto.

**¿Por qué primero ellos?** Todas las transformaciones posteriores (conversión de tipos, 
detección de outliers, análisis estadístico) dependen de que los nulos sean `NaN` real. 
Si dejamos "Sin información" como texto, `pd.to_numeric()` fallaría y `df.describe()` 
los ignoraría silenciosamente, produciendo estadísticas incorrectas.

In [3]:
def estandarizar_nulos(df):
    """
    Esta def reemplaza nulos disfrazados, por NaN real.
    
    Los valores tratados como nulo son:
    - Strings vacíos o que solo contienen espacios
    - "Sin información" 
    - "No aplica" 
    
    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con nulos disfrazados.
    
    Retorna
    -------
    pd.DataFrame
        Copia del DataFrame con nulos disfrazados convertidos a NaN.
    """
    df = df.copy()
    
    # Valores que representan ausencia de dato
    valores_nulos = {'sin información', 'no aplica'}
    
    # Recorrer solo columnas de texto
    cols_texto = df.select_dtypes(include='str').columns
    
    for col in cols_texto:
        # Paso 1: Limpiar espacios extremos
        df[col] = df[col].str.strip()
        
        # Paso 2: Strings vacíos se convierten en NaN
        df[col] = df[col].replace('', pd.NA)
        
        # Paso 3: Valores de la lista se convierten en NaN
        mascara = df[col].str.lower().isin(valores_nulos)
        df.loc[mascara, col] = pd.NA
    
    return df

In [4]:
# Aplicar T1
df = estandarizar_nulos(df_raw)

# Verificar: ¿cuántos NaN reales hay ahora?
nulos_despues = df.isnull().sum().sum()
nulos_esperados = 313_117 + 56_186 + 382  # Problemas 1 + 2 + 3

print(f"Nulos reales antes:    {estado_inicial['nulos_reales']:,}")
print(f"Nulos reales después:  {nulos_despues:,}")
print(f"Nulos esperados:       {nulos_esperados:,}")
print(f"Diferencia:            {nulos_despues - nulos_esperados:,}")

Nulos reales antes:    0
Nulos reales después:  369,685
Nulos esperados:       369,685
Diferencia:            0


### Verificación de $T_1$: ✅ Aprobada

Si la diferencia es **0**, capturamos exactamente los nulos disfrazados que identificamos 
en la auditoría. Si es **positiva**, hay valores disfrazados adicionales que no habíamos 
contado (variantes de capitalización, por ejemplo). Si es **negativa**, algo en la 
transformación no funcionó como esperábamos.

Así, obtuvimos que en efecto, la diferencia entre nulos encontrados y nulos esperados es **0**, lo que confirma 
que la función capturó exactamente los valores identificados en la auditoría, es decir, no se 
escapó ningún nulo disfrazado ni se convirtió ningún dato legítimo por error.

Por tanto, ahora `df.isnull()` refleja el estado real de completitud del dataset.

## 3. Fechas imposibles en `fecha_nac` ($T_2$)

**Problemas que resuelve:** #4 (fechas `9999-99-99` — 103 registros) y #5 (fechas 
parciales `año-99-99` — 98 registros).

**Estrategia:** Reemplazar por `NaN` toda fecha en `fecha_nac` que contenga `99` en la 
posición de mes o día. La información parcial del año no se pierde porque ya existe en 
la columna separada `anio_nac`.

**Criterio:** Una fecha es imposible si su mes es 99 o su día es 99. Formalmente:

$$\text{fecha imposible}(x) \iff x \text{ contiene el patrón } \texttt{-99-99} \text{ ó } \texttt{-99-}$$

Después de limpiar, convertimos `fecha_nac` de texto a `datetime`, lo cual permite 
calcular edades, rangos temporales y validaciones cruzadas con `fecha_def`.

In [5]:
def limpiar_fechas_nacimiento(df):
    """
    Convierte fechas imposibles en fecha_nac a NaN y transforma la columna a datetime.
    
    Fechas tratadas como imposibles:
    - '9999-99-99': fecha completamente desconocida
    - 'año-99-99': solo se conoce el año (ej: '1945-99-99')
    
    La información del año se preserva en la columna anio_nac.
    
    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con fecha_nac como texto.
    
    Retorna
    -------
    pd.DataFrame
        Copia con fecha_nac limpia y convertida a datetime.
    """
    df = df.copy()
    
    # Detectar fechas que contienen -99- en mes o día
    mascara_imposible = df['fecha_nac'].str.contains('-99-', na=False)
    
    fechas_imposibles = mascara_imposible.sum()
    print(f"Fechas imposibles detectadas en fecha_nac: {fechas_imposibles:,}")
    
    # Reemplazar por NaN
    df.loc[mascara_imposible, 'fecha_nac'] = pd.NA
    
    # Convertir a datetime (errors='coerce' convierte cualquier formato raro restante a NaT, equivalente de NaN para fechas)
    df['fecha_nac'] = pd.to_datetime(df['fecha_nac'], errors='coerce')
    
    return df

In [6]:
# Aplicar T2
df = limpiar_fechas_nacimiento(df)

# Verificar
fechas_esperadas = 103 + 98  # Problema #4 + Problema #5
fechas_nat = df['fecha_nac'].isna().sum()

# Contar cuántos NaN ya venían de T1 (nulos disfrazados en fecha_nac)
nulos_previos_fecha = df_raw['fecha_nac'].str.strip().isin(['', 'Sin información', 'No aplica']).sum()

fechas_nuevas_nat = fechas_nat - nulos_previos_fecha

print(f"NaT totales en fecha_nac:          {fechas_nat:,}")
print(f"  - NaN que ya venían de T1:       {nulos_previos_fecha:,}")
print(f"  - NaT nuevos por fechas imposibles: {fechas_nuevas_nat:,}")
print(f"  - Esperados (103 + 98):          {fechas_esperadas:,}")
print(f"  - Diferencia:                    {fechas_nuevas_nat - fechas_esperadas:,}")
print(f"\ndtype de fecha_nac ahora: {df['fecha_nac'].dtype}")

Fechas imposibles detectadas en fecha_nac: 201
NaT totales en fecha_nac:          201
  - NaN que ya venían de T1:       0
  - NaT nuevos por fechas imposibles: 201
  - Esperados (103 + 98):          201
  - Diferencia:                    0

dtype de fecha_nac ahora: datetime64[us]


### Verificación de $T_2$: ✅ Aprobada

La transformación detectó y convirtió **201 fechas imposibles** a `NaT`:

- 103 registros con `9999-99-99` (fecha completamente desconocida)
- 98 registros con `año-99-99` (solo se conocía el año)

Ninguna de estas celdas contenía nulos disfrazados previos de $T_1$, los 201 `NaT` 
son exclusivamente por fechas imposibles. La columna `fecha_nac` ahora es `datetime64`, 
lo que habilita operaciones temporales (diferencias de fechas, extracción de año/mes, 
validaciones cruzadas con `fecha_def`, etc).

## 4. Corrección de tipos de datos ($T_3$)

**Problema que resuelve:** #6 — las columnas `anio_nac`, `dia_nac` y `edad` están 
almacenadas como texto (`str`) cuando deberían ser numéricas.

**Consecuencia del tipo incorrecto:** operaciones como `df['edad'] > 80` producen 
comparaciones lexicográficas en lugar de numéricas, lo que genera filtros silenciosamente 
incorrectos. Las funciones `describe()`, `mean()`, `median()` no operan sobre texto.

**Estrategia:** Convertir a `Int64` (entero nullable de pandas) usando `pd.to_numeric` 
con `errors='coerce'` para que cualquier valor no convertible se transforme en `NaN` 
en lugar de lanzar un error.

**¿Por qué `Int64` y no `int64`?**  
El tipo `int64` de NumPy no admite valores nulos, si la columna tiene un solo `NaN`, 
la conversión fallaría. Por su parte, `Int64` (con mayúscula) es el tipo entero nullable de pandas(admite tanto enteros como `NaN` en la misma columna).

In [7]:
def corregir_tipos_numericos(df):
    """
    Convierte columnas numéricas almacenadas como texto a Int64.
    
    Columnas afectadas: anio_nac, dia_nac, edad.
    Usa pd.to_numeric con errors='coerce' para convertir valores
    no numéricos a NaN en lugar de lanzar excepciones.
    
    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con columnas numéricas en formato texto.
    
    Retorna
    -------
    pd.DataFrame
        Copia con las columnas convertidas a Int64 (entero nullable).
    """
    df = df.copy()
    
    columnas_a_convertir = ['anio_nac', 'dia_nac', 'edad']
    
    for col in columnas_a_convertir:
        tipo_antes = df[col].dtype
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
        tipo_despues = df[col].dtype
        print(f"  {col}: {tipo_antes} → {tipo_despues}")
    
    return df

In [8]:
# Tipos antes de T3
print("Tipos ANTES de T3:")
for col in ['anio_nac', 'dia_nac', 'edad']:
    print(f"  {col}: {df[col].dtype}")

print()

# Aplicar T3
print("Convirtiendo:")
df = corregir_tipos_numericos(df)

# Verificar que las estadísticas descriptivas ahora funcionan
print("\nEstadísticas descriptivas (solo posibles con tipos numéricos):")
print(df[['anio_nac', 'dia_nac', 'edad']].describe().round(1).to_string())

Tipos ANTES de T3:
  anio_nac: str
  dia_nac: str
  edad: str

Convirtiendo:
  anio_nac: str → Int64
  dia_nac: str → Int64
  edad: str → Int64

Estadísticas descriptivas (solo posibles con tipos numéricos):
       anio_nac   dia_nac      edad
count  107545.0  107447.0  107598.0
mean     1954.6      15.4      65.8
std        23.2       8.8      22.7
min      1858.0       1.0       0.0
25%      1937.0       8.0      54.0
50%      1950.0      15.0      71.0
75%      1966.0      23.0      83.0
max      2021.0      31.0     120.0


### Verificación de $T_3$: ✅ Aprobada

Las 3 columnas se convirtieron de `str` a `Int64`. Las estadísticas descriptivas 
confirman que la conversión fue exitosa (aparecen `mean`, `std`, `min`, `max` — 
imposibles con texto).

**Observaciones del `describe()` para la sección de outliers:**

| Columna | Valor sospechoso | Interpretación |
|---|---|---|
| `anio_nac` | min = 1858 | Una persona nacida en 1858 tendría 163 años en 2021, imposible |
| `edad` | max = 120 | Posible pero extremadamente improbable, requiere verificación |
| `dia_nac` | rango 1–31 | Normal, sin anomalías |

**Nulos después de la conversión** (diferencia entre count y 107,648 filas):

| Columna | Valores válidos | Nulos | Fuente probable |
|---|---|---|---|
| `anio_nac` | 107,545 | 103 | Los 103 registros con `9999-99-99` (año desconocido) |
| `dia_nac` | 107,447 | 201 | Los 201 registros con fechas imposibles (día desconocido) |
| `edad` | 107,598 | 50 | Valores que eran nulos disfrazados en $T_1$ |

Estos hallazgos alimentarán la sección de detección de outliers ($T_7$).

## 5. Eliminación de duplicados ($T_4$)

**Problema que resuelve:** #9 — 7 pares de registros duplicados (14 filas) donde todas 
las columnas son idénticas excepto `Numeracion` (el identificador único asignado por 
el sistema).

**Estrategia:** Eliminar la segunda ocurrencia de cada par, conservando la primera. 
La comparación se hace sobre todas las columnas excepto `Numeracion`, ya que este campo 
es un ID secuencial que siempre difiere entre registros. Si lo incluyéramos, ningún par 
sería detectado como duplicado.

**Decisión documentada:** Se eliminan (no se marcan) porque son duplicaciones exactas 
del mismo evento. No hay información adicional en la segunda copia que justifique 
conservarla.

In [9]:
def eliminar_duplicados(df):
    """
    Elimina filas duplicadas ignorando la columna Numeracion.
    
    Conserva la primera ocurrencia de cada par duplicado.
    La columna Numeracion se excluye de la comparación porque es un
    identificador único asignado por el sistema (siempre difiere).
    
    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame posiblemente con filas duplicadas.
    
    Retorna
    -------
    pd.DataFrame
        Copia sin duplicados, con índice reiniciado.
    """
    df = df.copy()
    
    # Columnas de comparación: todas excepto el ID
    cols_comparacion = [col for col in df.columns if col != 'Numeracion']
    
    filas_antes = len(df)
    df = df.drop_duplicates(subset=cols_comparacion, keep='first')
    filas_despues = len(df)
    
    eliminadas = filas_antes - filas_despues
    print(f"  Filas antes:      {filas_antes:,}")
    print(f"  Filas después:    {filas_despues:,}")
    print(f"  Filas eliminadas: {eliminadas:,}")
    
    # Reiniciar índice para evitar huecos. Sin esto, el DataFrame conserva los índices originales con "huecos" 
    # (ejm: 0, 1, 2, 4, 5 — falta el 3 porque era duplicado)
    df = df.reset_index(drop=True)
    
    return df

In [10]:
# Aplicar T4
print("Eliminando duplicados (ignorando Numeracion):")
df = eliminar_duplicados(df)

# Verificar
duplicados_restantes = df.drop(columns='Numeracion').duplicated().sum()
print(f"\n  Duplicados restantes: {duplicados_restantes}")
print(f"  Filas esperadas: {107_648 - 7:,}")
print(f"  Filas actuales:  {len(df):,}")

Eliminando duplicados (ignorando Numeracion):
  Filas antes:      107,648
  Filas después:    107,641
  Filas eliminadas: 7

  Duplicados restantes: 0
  Filas esperadas: 107,641
  Filas actuales:  107,641


### Verificación de $T_4$: ✅ Aprobada

Se eliminaron exactamente **7 filas** (la segunda copia de cada par duplicado):

- Filas antes: 107,648 → Filas después: 107,641
- Duplicados restantes: 0

El dataset pasa de 107,648 a **107,641 registros** — esta es la nueva línea base 
para el resto del pipeline.

## 6. Clasificación de registros tardíos ($T_5$)

**Problema que resuelve:** #7 — 2,400 registros donde `anio_fall` ≠ 2021. Son 
defunciones ocurridas en años anteriores (rango 1900–2020, mayoría en 2020) pero 

**Debate:** La pregunta es: ¿qué hacemos con ellos?

| Opción | Ventaja | Desventaja |
|---|---|---|
| **Eliminarlos** | Dataset puro de defunciones 2021 | Se pierde 2,400 registros (2.2% del total) |
| **Marcarlos** con una columna flag | No pierdes datos, puedes filtrar después | Se añade complejidad al análisis |
| **No hacer nada** | Simplicidad | Las estadísticas de 2021 están contaminadas con datos de otros años |

**Decisión:** No se eliminan. Se crea una columna flag `es_registro_tardio` que permite 
filtrarlos en análisis posteriores sin perder información.

**Justificación:** En datos administrativos, los registros tardíos son un fenómeno 
esperado, no un error de calidad. Un Data Analyst que reciba este dataset limpio puede 
decidir incluirlos o excluirlos según su pregunta de análisis, sin que el pipeline haya 
tomado esa decisión irreversiblemente por él.

In [11]:
def clasificar_registros_tardios(df):
    """
    Agrega columna flag para identificar registros con anio_fall != 2021.
    
    Los registros tardíos son defunciones ocurridas antes de 2021 pero 
    registradas oficialmente en ese año. No se eliminan, se marcan 
    para que el analista (de ser el caso) pueda filtrar según su necesidad.
    
    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con columna anio_fall.
    
    Retorna
    -------
    pd.DataFrame
        Copia con nueva columna 'es_registro_tardio' (bool).
    """
    df = df.copy()
    
    df['es_registro_tardio'] = df['anio_fall'] != 2021
    
    tardios = df['es_registro_tardio'].sum()
    print(f"  Registros tardíos (anio_fall != 2021): {tardios:,}")
    print(f"  Registros de 2021:                     {(~df['es_registro_tardio']).sum():,}")
    
    return df

In [12]:
# Aplicar T5
print("Clasificando registros tardíos:")
df = clasificar_registros_tardios(df)

# Verificar distribución por año
print(f"\nDistribución de anio_fall:")
print(df['anio_fall'].value_counts().head(10).to_string())
print(f"\nColumnas del DataFrame: {df.shape[1]} (era 45, ahora debe ser 46)")

Clasificando registros tardíos:
  Registros tardíos (anio_fall != 2021): 2,399
  Registros de 2021:                     105,242

Distribución de anio_fall:
anio_fall
2021    105242
2020      1684
2019        91
2018        51
2016        46
2017        33
2015        29
2014        28
2007        20
2008        20

Columnas del DataFrame: 46 (era 45, ahora debe ser 46)


### Verificación de $T_5$: ✅ Aprobada

Se identificaron **2,399 registros tardíos** (defunciones con `anio_fall` ≠ 2021).

La auditoría original reportó 2,400, pero uno de los 7 duplicados eliminados en $T_4$ 
era un registro tardío — al remover la copia, el conteo bajó en 1. Los números son 
consistentes.

**Distribución:** la mayoría de registros tardíos provienen de 2020 (1,684 — 
probablemente trámites que cruzaron el cambio de año). El registro más antiguo data 
de 1900, lo que sugiere regularizaciones administrativas de décadas anteriores.

Se creó la columna `es_registro_tardio` (boolean). El DataFrame tiene ahora **46 columnas** 
(45 originales + 1 flag).

## 7. Separar código y descripción en columnas de causa ($T_6$)

**Problema que resuelve:** #8 — 6 columnas de causa de muerte tienen el código 
clasificatorio y la descripción concatenados en una sola celda 
(ejemplo: `I10   Hipertensión esencial (primaria)`).

Con el formato actual, agrupar por código de causa requiere manipulación de strings 
sobre texto libre, el cual es un enfoque frágil y propenso a errores. Separar código y 
descripción en columnas independientes permite operaciones directas como 
`df.groupby('cod_causa')`.

### Exploración de patrones

Las 6 columnas presentan tres patrones de formato distintos:

| Formato | Columnas | Ejemplo | Registros |
|---|---|---|---|
| CIE-10 (letra + dígitos) | `causa4`, `causa` | `I10   Hipertensión esencial (primaria)` | 86,639 – 107,641 |
| Numérico (3 dígitos) | `causa103`, `causa80`, `causa67B` | `066 Enfermedades hipertensivas` | 107,450 – 107,641 |
| Mixto (ambos formatos) | `causa67A` | CIE-10 en 21,002 + numérico en 86,448 | 107,450 |

El rango 86,639 – 107,641 intenta decir "la columna con menos registros de ese patrón tiene 86,639 y la que más tiene 107,641". No es un rango continuo, son conteos de columnas distintas.

**Caso especial:** `causa4` contiene 21,002 registros (todos COVID-19) donde no existe 
código separado — el campo es solo texto descriptivo: `COVID-19, virus no identificado`. 
Estos mismos registros sí tienen código `U07` en la columna `causa`, por lo que la 
información no se pierde.

**Debate:** La pregunta es ¿qué estrategia de separación usamos?

| Opción | Estrategia | Ventaja | Desventaja |
|---|---|---|---|
| **Regex por tipo de columna** | Un patrón para CIE-10 y otro para numéricos | Preciso, respeta la estructura de cada columna | Más líneas de código |
| **Regex universal** | Un solo patrón para todas | Menos código | No captura los 21,002 COVID sin código en `causa4` |
| **Split por espacios** | Separar en el primer bloque de espacios | Simple | Falla con formatos de un solo espacio (`066 Enfermedades...`) |

**Decisión:** Regex por tipo de columna. Cada tipo de columna tiene una estructura definida y estable. 
Usar un regex específico por tipo garantiza que la extracción sea correcta incluso para 
los casos atípicos (COVID sin código en `causa4`, formato mixto en `causa67A`). Cuando 
el código no existe, se asigna `NaN` — no se inventa.

In [ ]:
def separar_codigo_descripcion(df):
    """
    Separa código y descripción en las 6 columnas de causa de muerte.
    
    Genera dos columnas nuevas por cada original: 'cod_[nombre]' y 'desc_[nombre]'.
    Las columnas originales se conservan sin modificación.
    
    Patrones reconocidos:
    - CIE-10: letra seguida de 2-3 dígitos (ej: 'I10', 'U072', 'G809')
    - Numérico: 3 dígitos al inicio (ej: '066', '104')
    
    Cuando no se detecta código, se asigna NaN al código y el texto 
    completo a la descripción.
    
    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con columnas de causa concatenadas.
    
    Retorna
    -------
    pd.DataFrame
        Copia con 12 columnas nuevas (2 por cada columna de causa).
    """
    import re
    
    df = df.copy()
    
    # Clasificación de columnas por tipo de formato
    cols_cie10 = ['causa4', 'causa']
    cols_numericas = ['causa103', 'causa80', 'causa67B']
    cols_mixtas = ['causa67A']
    
    def extraer_cie10(valor):
        """
        Extrae código CIE-10 y descripción de una celda.
        
        Busca el patrón: letra + 2-3 dígitos (opcionalmente seguidos 
        de punto y más dígitos) y espacios, seguido por la descripción.
        
        Parámetros
        ----------
        valor : str o NaN
            Valor de la celda con código y descripción concatenados.
        
        Retorna
        -------
        tuple (código, descripción)
            Si se detecta código: (código_extraído, descripción)
            Si no: (NaN, valor_completo)
        """
        if pd.isna(valor):
            return pd.NA, pd.NA
        match = re.match(r'^([A-Za-z]\d{2,3}(?:\.\d+)?)\s+(.*)', str(valor))
        if match:
            return match.group(1).strip(), match.group(2).strip()
        return pd.NA, str(valor).strip()
    
    def extraer_numerico(valor):
        """
        Extrae código numérico de 3 dígitos y descripción.
        
        Busca 3 dígitos al inicio, seguidos de espacios y descripción.
        
        Parámetros
        ----------
        valor : str o NaN
            Valor de la celda.
        
        Retorna
        -------
        tuple (código, descripción)
            Si se detectan 3 dígitos: (código_extraído, descripción)
            Si no: (NaN, valor_completo)
        """
        if pd.isna(valor):
            return pd.NA, pd.NA
        match = re.match(r'^(\d{3})\s+(.*)', str(valor))
        if match:
            return match.group(1).strip(), match.group(2).strip()
        return pd.NA, str(valor).strip()
    
    def extraer_mixto(valor):
        """
        Extrae código (CIE-10 o numérico) y descripción de formato mixto.
        
        Intenta primero el patrón CIE-10; si falla, intenta numérico.
        Permite manejar celdas que pueden tener cualquiera de los dos formatos.
        
        Parámetros
        ----------
        valor : str o NaN
            Valor de la celda.
        
        Retorna
        -------
        tuple (código, descripción)
            Código y descripción extraídos, o (NaN, valor_completo) si no coincide.
        """
        if pd.isna(valor):
            return pd.NA, pd.NA
        # Intentar CIE-10 primero
        match = re.match(r'^([A-Za-z]\d{2,3}(?:\.\d+)?)\s+(.*)', str(valor))
        if match:
            return match.group(1).strip(), match.group(2).strip()
        # Luego intentar numérico
        match = re.match(r'^(\d{3})\s+(.*)', str(valor))
        if match:
            return match.group(1).strip(), match.group(2).strip()
        return pd.NA, str(valor).strip()
    
    # Mapeo de columnas a su función extractora
    mapeo = {}
    for col in cols_cie10:
        mapeo[col] = extraer_cie10
    for col in cols_numericas:
        mapeo[col] = extraer_numerico
    for col in cols_mixtas:
        mapeo[col] = extraer_mixto
    
    # Aplicar extracción
    for col, funcion in mapeo.items():
        resultados = df[col].apply(funcion)
        df[f'cod_{col}'] = resultados.apply(lambda x: x[0])
        df[f'desc_{col}'] = resultados.apply(lambda x: x[1])
        
        # Reporte
        sin_codigo = df[f'cod_{col}'].isna().sum() - df[col].isna().sum()
        print(f"  {col}: {sin_codigo:,} registros sin código extraíble")
    
    return df

In [14]:
# Aplicar T6
print("Separando código y descripción en columnas de causa:")
df = separar_codigo_descripcion(df)

# Verificar: mostrar ejemplos de la separación
print(f"\nColumnas del DataFrame: {df.shape[1]} (46 + 12 nuevas = 58 esperadas)")

# Muestra comparativa de la columna más limpia (causa)
print("\nEjemplo de separación en 'causa':")
muestra = df[['causa', 'cod_causa', 'desc_causa']].dropna().head(5)
print(muestra.to_string(index=False))

# Muestra del caso especial COVID en causa4
print("\nEjemplo del caso COVID en 'causa4' (sin código CIE-10):")
covid_mask = df['cod_causa4'].isna() & df['causa4'].notna()
if covid_mask.sum() > 0:
    muestra_covid = df.loc[covid_mask, ['causa4', 'cod_causa4', 'desc_causa4']].head(3)
    print(muestra_covid.to_string(index=False))

Separando código y descripción en columnas de causa:
  causa4: 21,002 registros sin código extraíble
  causa: 0 registros sin código extraíble
  causa103: 0 registros sin código extraíble
  causa80: 1 registros sin código extraíble
  causa67B: 0 registros sin código extraíble
  causa67A: 0 registros sin código extraíble

Columnas del DataFrame: 58 (46 + 12 nuevas = 58 esperadas)

Ejemplo de separación en 'causa':
                                   causa cod_causa                         desc_causa
  I10   Hipertensión esencial (primaria)       I10   Hipertensión esencial (primaria)
       I21   Infarto agudo del miocardio       I21        Infarto agudo del miocardio
U07   COVID-19 Confirmados y sospechosos       U07 COVID-19 Confirmados y sospechosos
             R98   Muerte sin asistencia       R98              Muerte sin asistencia
                G80   Parálisis cerebral       G80                 Parálisis cerebral

Ejemplo del caso COVID en 'causa4' (sin código CIE-10):
          

### Verificación de $T_6$: ✅ Aprobada

La extracción generó 12 columnas nuevas (2 por cada columna de causa). El DataFrame 
tiene ahora **58 columnas** (46 previas + 12 nuevas).

**Resultados por columna:**

| Columna | Registros con código | Sin código | Observación |
|---|---|---|---|
| `causa4` | 86,639 | 21,002 | Los 21,002 son registros COVID, su código `U07` sí existe en `causa` |
| `causa` | 107,641 | 0 | Extracción completa |
| `causa103` | 107,641 | 0 | Extracción completa |
| `causa80` | 107,640 | 1 | 1 registro con formato no reconocido |
| `causa67A` | 107,450 | 0 | Formato mixto manejado correctamente |
| `causa67B` | 107,450 | 0 | Extracción completa |

El caso COVID en `causa4` se resolvió según lo diseñado: código como `NaN`, 
descripción completa preservada. La información del código CIE-10 (`U07`) permanece 
accesible en `cod_causa`.

## 8. Detección de outliers con criterio matemático ($T_7$)

El análisis de las columnas numéricas requiere evaluar si existen valores extremos que 
representen errores de registro. Se aplican dos métodos formales — la regla del IQR y 
el z-score — no para eliminar automáticamente, sino para identificar candidatos que 
luego se evalúan con contexto del dominio.

### Fundamento matemático

**Regla del IQR (Rango Intercuartílico):**

Sea $Q_1$ el primer cuartil y $Q_3$ el tercer cuartil de una variable. El rango 
intercuartílico es $\text{IQR} = Q_3 - Q_1$. Un valor $x$ se considera outlier si:

$$x < Q_1 - 1.5 \cdot \text{IQR} \quad \text{ó} \quad x > Q_3 + 1.5 \cdot \text{IQR}$$

Este criterio se basa en la dispersión central del 50% de los datos. El factor 1.5 es 
una convención propuesta por Tukey (1977) — para una distribución normal, captura 
aproximadamente el 99.3% de las observaciones.

**Z-score:**

El z-score mide cuántas desviaciones estándar se aleja un valor de la media:

$$z_i = \frac{x_i - \bar{x}}{s}$$

Un valor se considera outlier si $|z_i| > 3$, lo que en una distribución normal 
corresponde al 0.27% de las observaciones más extremas.

### Debate: ¿qué hacer con los outliers detectados?

| Opción | Cuándo aplicar | Riesgo |
|---|---|---|
| **Eliminar** | El valor es un error verificable | Se pierde información real |
| **Reemplazar** (winsorización, imputación) | El valor es extremo pero la observación importa | Se distorsiona la distribución original |
| **Marcar y conservar** | El valor es extremo pero legítimo | Complejidad adicional en análisis |
| **No hacer nada** | Todos los valores extremos son legítimos | Sesgo en estadísticas de resumen |

### Resultados de la detección

Las columnas numéricas del dataset son: `Numeracion`, `anio_nac`, `dia_nac`, `anio_fall`, 
`dia_fall` y `edad`. De estas, `Numeracion` es un identificador secuencial (no tiene 
sentido buscar outliers) y `dia_nac`/`dia_fall` tienen rango fijo [1, 31] sin outliers.

Las tres columnas con outliers según IQR:

| Columna | Outliers IQR | Interpretación | Decisión |
|---|---|---|---|
| `anio_fall` < 2021 | 2,399 | Registros tardíos, ya marcados con flag en $T_5$ | **No hacer nada** — ya tratados |
| `edad` < 10.5 | 3,331 | Defunciones infantiles y de niños — dato demográfico legítimo | **No hacer nada** — son datos reales |
| `anio_nac` < 1894 | 16 | Registros tardíos extremos — validación cruzada con `edad` y `anio_fall` confirma consistencia interna | **No hacer nada** — internamente consistentes |
| `anio_nac` > 2009 | 3,886 | Niños nacidos después de 2009, consistente con las 3,331 muertes infantiles en `edad` | **No hacer nada** — dato demográfico legítimo |

**Conclusión: no se eliminan ni modifican valores.** Todos los outliers estadísticos 
son datos legítimos en el contexto de un registro de defunciones. Esta conclusión 
se respaldó mediante validación cruzada entre variables (`anio_fall - anio_nac ≈ edad`) 
y conocimiento del dominio (las muertes infantiles son un fenómeno demográfico real, 
no un error de registro).

> **Nota metodológica:** Outlier estadístico ≠ error de datos. El IQR y el z-score 
> son criterios matemáticos ciegos al contexto. El analista aporta el contexto que 
> determina si un valor extremo es un error o un dato legítimo.

In [15]:
def detectar_outliers(df):
    """
    Aplica detección de outliers con IQR y z-score a columnas numéricas relevantes.
    
    No modifica el DataFrame. Genera un reporte comparativo de ambos métodos
    para documentar que los outliers detectados fueron evaluados y clasificados 
    como datos legítimos.
    
    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con columnas numéricas.
    
    Retorna
    -------
    pd.DataFrame
        Resumen de outliers detectados por columna y método.
    """
    columnas_analizar = ['anio_nac', 'edad', 'anio_fall']
    resultados = []
    
    for col in columnas_analizar:
        valores = df[col].dropna()
        
        # Método IQR
        q1 = valores.quantile(0.25)
        q3 = valores.quantile(0.75)
        iqr = q3 - q1
        lim_inf_iqr = q1 - 1.5 * iqr
        lim_sup_iqr = q3 + 1.5 * iqr
        outliers_iqr = ((valores < lim_inf_iqr) | (valores > lim_sup_iqr)).sum()
        
        # Método z-score
        media = valores.mean()
        std = valores.std()
        z_scores = ((valores - media) / std).abs()
        outliers_z = (z_scores > 3).sum()
        
        resultados.append({
            'columna': col,
            'Q1': q1,
            'Q3': q3,
            'IQR': iqr,
            'lim_inf_iqr': lim_inf_iqr,
            'lim_sup_iqr': lim_sup_iqr,
            'outliers_iqr': outliers_iqr,
            'media': round(media, 1),
            'std': round(std, 1),
            'outliers_zscore': outliers_z
        })
    
    resumen = pd.DataFrame(resultados)
    return resumen


# Aplicar detección
resumen_outliers = detectar_outliers(df)
print("Comparación de métodos de detección de outliers:\n")
print(resumen_outliers.to_string(index=False))

Comparación de métodos de detección de outliers:

  columna     Q1     Q3  IQR  lim_inf_iqr  lim_sup_iqr  outliers_iqr  media  std  outliers_zscore
 anio_nac 1937.0 1966.0 29.0       1893.5       2009.5          3902 1954.6 23.1               12
     edad   54.0   83.0 29.0         10.5        126.5          3331   65.8 22.7                0
anio_fall 2021.0 2021.0  0.0       2021.0       2021.0          2399 2020.9  2.2              465


### Verificación de $T_7$: ✅ Completada — sin modificaciones al dataset

**Comparación entre métodos:**

| Columna | Outliers IQR | Outliers z-score | ¿Por qué difieren? |
|---|---|---|---|
| `anio_nac` | 3,902 | 12 | La distribución es asimétrica (más muertes infantiles que centenarios) — IQR es más sensible a la asimetría |
| `edad` | 3,331 | 0 | El rango [0, 120] cabe dentro de $\bar{x} \pm 3s$ = [−2.3, 133.9] — z-score no detecta nada |
| `anio_fall` | 2,399 | 465 | El IQR es 0 (Q1 = Q3 = 2021), así que cualquier valor ≠ 2021 es outlier por IQR. Z-score solo marca los más extremos |

**Observación metodológica:** Los dos métodos no son intercambiables. El IQR no asume 
normalidad y es robusto ante asimetría, pero cuando la distribución está muy concentrada 
(como `anio_fall`, donde el 97.8% de los valores son 2021), el IQR colapsa a 0 y marca 
todo lo demás como outlier. El z-score asume normalidad, lo que lo hace inadecuado para 
distribuciones fuertemente sesgadas como `edad` en un registro de defunciones.

**Decisión final:** No se eliminan ni modifican valores en ninguna columna. La 
validación cruzada entre `anio_nac`, `anio_fall` y `edad` confirmó consistencia 
interna incluso en los 16 registros más extremos (nacidos antes de 1894). Todos los 
outliers estadísticos corresponden a fenómenos demográficos reales: muertes infantiles, 
registros tardíos y regularizaciones administrativas históricas.

## 9. Función `limpiar_dataset(df)`, la composición del pipeline

Las transformaciones $T_1$ a $T_6$ se aplicaron individualmente para poder verificar 
cada paso. Ahora se encapsulan en una sola función que ejecuta el pipeline completo 
sobre el dataset crudo, produciendo el dataset limpio de forma reproducible.

$$\text{limpiar\_dataset} = T_6 \circ T_5 \circ T_4 \circ T_3 \circ T_2 \circ T_1$$

**¿Por qué $T_7$ (detección de outliers) no está en la composición?**

Una transformación se incluye en la composición funcional si **modifica el DataFrame**. 
Las transformaciones $T_1$ a $T_6$ modifican datos (reemplazan nulos, eliminan filas, 
crean columnas, etc.). Por el contrario, $T_7$ es un **análisis exploratorio** que 
genera un resumen comparativo de dos métodos de detección sin alterar el DataFrame. 
Su resultado fue: "todos los outliers son datos legítimos, no se modifica nada". 

En términos funcionales, una función de análisis que no modifica su entrada no pertenece 
a la cadena de transformaciones puras. Se mantiene documentada como análisis ejecutado 
y verificado, pero el pipeline de limpieza es $T_6 \circ \cdots \circ T_1$.

Para verificar que la función compuesta produce el mismo resultado que la aplicación 
secuencial, se compara el DataFrame resultante contra el `df` que se construyó paso 
a paso a lo largo del notebook.

In [16]:
def limpiar_dataset(df_crudo):
    """
    Pipeline completo de limpieza del dataset de Defunciones Generales Ecuador 2021.
    
    Aplica 6 transformaciones en orden, respetando las dependencias entre ellas:
    T1: Estandarización de nulos disfrazados (espacios, "Sin información", "No aplica")
    T2: Limpieza de fechas imposibles en fecha_nac
    T3: Corrección de tipos numéricos (anio_nac, dia_nac, edad)
    T4: Eliminación de duplicados (ignorando Numeracion)
    T5: Clasificación de registros tardíos (flag es_registro_tardio)
    T6: Separación de código y descripción en columnas de causa
    
    Parámetros
    ----------
    df_crudo : pd.DataFrame
        Dataset crudo tal como se carga desde el CSV (sep=';').
    
    Retorna
    -------
    pd.DataFrame
        Dataset limpio con nulos reales, tipos correctos, sin duplicados,
        registros tardíos marcados y columnas de causa separadas.
    """
    df = (
        df_crudo
        .pipe(estandarizar_nulos)          # T1: nulos disfrazados → NaN
        .pipe(limpiar_fechas_nacimiento)    # T2: fechas imposibles → NaT
        .pipe(corregir_tipos_numericos)     # T3: str → Int64
        .pipe(eliminar_duplicados)          # T4: 7 duplicados eliminados
        .pipe(clasificar_registros_tardios) # T5: flag es_registro_tardio
        .pipe(separar_codigo_descripcion)   # T6: código + descripción → 2 columnas
    )
    
    return df

In [17]:
# Ejecutar el pipeline completo desde el dataset crudo
df_verificacion = limpiar_dataset(df_raw)

# Comparar contra el df construido paso a paso
misma_forma = df_verificacion.shape == df.shape

# Comparación columna por columna (respeta tipos mixtos)
columnas_iguales = 0
columnas_diferentes = []

for col in df.columns:
    # Comparar considerando NaN == NaN (por defecto NaN != NaN)
    iguales = df[col].equals(df_verificacion[col])
    if iguales:
        columnas_iguales += 1
    else:
        columnas_diferentes.append(col)
        
print("Verificación de reproducibilidad:")
print(f"  Misma forma (shape):    {df_verificacion.shape} == {df.shape} → {misma_forma}")
print(f"  Columnas idénticas:     {columnas_iguales} de {len(df.columns)}")

if columnas_diferentes:
    print(f"  Columnas diferentes:    {columnas_diferentes}")
else:
    print(f"\n  En efecto, la función limpiar_dataset() reproduce exactamente el resultado")
    print(f"     del pipeline aplicado paso a paso.")

Fechas imposibles detectadas en fecha_nac: 201
  anio_nac: str → Int64
  dia_nac: str → Int64
  edad: str → Int64
  Filas antes:      107,648
  Filas después:    107,641
  Filas eliminadas: 7
  Registros tardíos (anio_fall != 2021): 2,399
  Registros de 2021:                     105,242
  causa4: 21,002 registros sin código extraíble
  causa: 0 registros sin código extraíble
  causa103: 0 registros sin código extraíble
  causa80: 1 registros sin código extraíble
  causa67B: 0 registros sin código extraíble
  causa67A: 0 registros sin código extraíble
Verificación de reproducibilidad:
  Misma forma (shape):    (107641, 58) == (107641, 58) → True
  Columnas idénticas:     58 de 58

  En efecto, la función limpiar_dataset() reproduce exactamente el resultado
     del pipeline aplicado paso a paso.


### Verificación de la composición: ✅ Aprobada

La función `limpiar_dataset()` reproduce exactamente el resultado del pipeline 
aplicado paso a paso: misma forma (107,641 × 58) y 58 de 58 columnas idénticas.

Esto confirma que las 6 transformaciones son funciones puras sin dependencias ocultas 
ni efectos secundarios. Así, la composición $T_6 \circ T_5 \circ \cdots \circ T_1$ es 
determinista y reproducible.

## 10. Comparación antes/después y exportación del dataset limpio

Resumen cuantitativo del impacto del pipeline de limpieza sobre el dataset.

In [18]:
# Estado final del dataset limpio
estado_final = {
    'filas': df.shape[0],
    'columnas': df.shape[1],
    'nulos_reales': df.isnull().sum().sum(),
    'duplicados_exactos': df.drop(columns='Numeracion').duplicated().sum(),
    'tipos_texto': (df.dtypes == 'str').sum(),
    'memoria_mb': df.memory_usage(deep=True).sum() / 1_048_576
}

# Construir tabla comparativa
comparacion = pd.DataFrame({
    'Métrica': [
        'Filas',
        'Columnas',
        'Nulos reales (NaN/NaT)',
        'Duplicados exactos',
        'Columnas tipo texto',
        'Memoria (MB)'
    ],
    'Antes (crudo)': [
        f"{estado_inicial['filas']:,}",
        f"{estado_inicial['columnas']}",
        f"{estado_inicial['nulos_reales']:,}",
        f"{estado_inicial['duplicados_exactos']}",
        f"{estado_inicial['tipos_texto']}",
        f"{estado_inicial['memoria_mb']:.2f}"
    ],
    'Después (limpio)': [
        f"{estado_final['filas']:,}",
        f"{estado_final['columnas']}",
        f"{estado_final['nulos_reales']:,}",
        f"{estado_final['duplicados_exactos']}",
        f"{estado_final['tipos_texto']}",
        f"{estado_final['memoria_mb']:.2f}"
    ],
    'Cambio': [
        f"{estado_final['filas'] - estado_inicial['filas']:,}",
        f"+{estado_final['columnas'] - estado_inicial['columnas']}",
        f"+{estado_final['nulos_reales'] - estado_inicial['nulos_reales']:,}",
        f"{estado_final['duplicados_exactos'] - estado_inicial['duplicados_exactos']}",
        f"{estado_final['tipos_texto'] - estado_inicial['tipos_texto']}",
        f"{estado_final['memoria_mb'] - estado_inicial['memoria_mb']:.2f}"
    ]
})

print("Comparación antes/después del pipeline de limpieza:\n")
print(comparacion.to_string(index=False))

Comparación antes/después del pipeline de limpieza:

               Métrica Antes (crudo) Después (limpio)   Cambio
                 Filas       107,648          107,641       -7
              Columnas            45               58      +13
Nulos reales (NaN/NaT)             0          391,605 +391,605
    Duplicados exactos             7                0       -7
   Columnas tipo texto            42               50        8
          Memoria (MB)        336.30           408.29    71.99


### Resumen del pipeline

| Métrica | Impacto | Causa |
|---|---|---|
| -7 filas | 107,648 → 107,641 | 7 duplicados exactos eliminados |
| +13 columnas | 45 → 58 | 12 de separación código/descripción + 1 flag de registro tardío |
| +391,605 nulos reales | 0 → 391,605 | Nulos disfrazados revelados ($T_1$: 369,685) + fechas imposibles ($T_2$: 201) + NaN por conversión de tipos ($T_3$) + códigos no extraíbles ($T_6$) |
| -7 duplicados | 7 → 0 | Todos los pares eliminados |
| +8 columnas texto | 42 → 50 | Las 12 nuevas columnas `desc_` y `cod_` son texto |
| +72 MB memoria | 336 → 408 MB | Columnas adicionales y conversión de tipos |

**Observación clave:** El dataset crudo reportaba 0 nulos. El pipeline reveló que 
en realidad existían **391,605 celdas sin dato válido** — un 8.1% del total de celdas 
del dataset original (107,648 × 45 = 4,844,160). La limpieza no "ensució" el dataset; 
lo hizo honesto.

El dataset limpio se exporta a `data/processed/defunciones_2021_limpio.csv`.

In [19]:
# Exportar dataset limpio a data/processed/
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

archivo_salida = PROCESSED_PATH / 'defunciones_2021_limpio.csv'
df.to_csv(archivo_salida, index=False)

print(f"Dataset limpio exportado a: {archivo_salida}")
print(f"  Filas: {len(df):,}")
print(f"  Columnas: {df.shape[1]}")
print(f"  Tamaño: {archivo_salida.stat().st_size / 1_048_576:.2f} MB")

Dataset limpio exportado a: ..\data\processed\defunciones_2021_limpio.csv
  Filas: 107,641
  Columnas: 58
  Tamaño: 85.53 MB


### Siguiente paso

En el Notebook 03 se cargará el dataset limpio exportado y se realizará el análisis 
exploratorio completo: distribuciones de variables numéricas, consultas SQL sobre el 
dataset cargado en SQLite, matriz de correlación, y un mínimo de 8 visualizaciones — 
cada una con interpretación. Nuestro objetivo ahora es extraer las historias que los datos cuentan 
una vez que son confiables.